In [3]:
import pandas as pd
import glob
import numpy as np
from datetime import timedelta
from dateutil.relativedelta import relativedelta

# ================= CONFIG =================
FOLDER = r"C:\MyDocuments\03Business\05ResearchAndAnalysis\StockInvestments\ResearchReports\YahooConsensus"
MIN_ANALYSTS = 5
TOP_N = 10

PERIODS = ["1W", "2W", "1M", "2M", "3M"]

# ================= LOAD DATA =================
files = sorted(glob.glob(f"{FOLDER}\\*_YahooConsensus.csv"))

df_all = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)
df_all["Date"] = pd.to_datetime(df_all["Date"])
df_all = df_all[df_all["#Count"] >= MIN_ANALYSTS].copy()

latest_date = df_all["Date"].max()
latest_date_str = latest_date.strftime("%Y%m%d")

output_file = f"{FOLDER}\\{latest_date_str}_YahooConsensusTrendAnalysis.xlsx"

# ================= DATE HELPERS =================
def get_period_target(latest, label):
    num = int(label[:-1])
    unit = label[-1]

    if unit == "W":
        return latest - timedelta(weeks=num)
    elif unit == "M":
        return latest - relativedelta(months=num)
    else:
        raise ValueError(f"Invalid period label: {label}")

def get_asof_date(dates, target_date):
    eligible = dates[dates <= target_date]
    return eligible.max() if not eligible.empty else None

# ================= TARGET PRICE REVISION =================
def compute_tp_revision(df, col, period_label):
    latest_date = df["Date"].max()
    target_date = get_period_target(latest_date, period_label)
    past_date = get_asof_date(df["Date"], target_date)

    if past_date is None:
        return pd.DataFrame()

    df_latest = df[df["Date"] == latest_date].drop_duplicates("YAHOO Ticker")
    df_past = df[df["Date"] == past_date].drop_duplicates("YAHOO Ticker")

    merged = df_latest.merge(df_past, on="YAHOO Ticker", suffixes=("_L", "_P"))

    merged = merged[(merged[f"{col}_P"] > 0) & (merged[f"{col}_L"] > 0)]
    if merged.empty:
        return pd.DataFrame()

    # TP Change
    merged["Change"] = merged[f"{col}_L"] / merged[f"{col}_P"] - 1

    # Price Change
    merged["Price_Change"] = merged["Current Price_L"] / merged["Current Price_P"] - 1

    # Clean inf/nan
    merged["Change"] = merged["Change"].replace([np.inf, -np.inf], 0).fillna(0)
    merged["Price_Change"] = merged["Price_Change"].replace([np.inf, -np.inf], 0).fillna(0)

    # Round to 4 decimals
    merged["Change"] = merged["Change"].round(4)
    merged["Price_Change"] = merged["Price_Change"].round(4)

    merged["#Count"] = merged["#Count_L"]
    merged["FIN Yr"] = ""  # TP has no FIN Yr

    return merged[["YAHOO Ticker", "FIN Yr", "Change", "Price_Change", "#Count"]]

# ================= REV/EPS REVISION =================
def compute_revision(df, col, period_label, fin_year):
    latest_date = df["Date"].max()
    target_date = get_period_target(latest_date, period_label)
    past_date = get_asof_date(df["Date"], target_date)

    if past_date is None:
        return pd.DataFrame()

    df_latest = df[(df["Date"] == latest_date) & (df["FIN Yr"] == fin_year)]
    df_past = df[(df["Date"] == past_date) & (df["FIN Yr"] == fin_year)]

    if df_latest.empty or df_past.empty:
        return pd.DataFrame()

    df_latest = df_latest.drop_duplicates(["YAHOO Ticker", "FIN Yr"])
    df_past = df_past.drop_duplicates(["YAHOO Ticker", "FIN Yr"])

    merged = df_latest.merge(df_past, on=["YAHOO Ticker", "FIN Yr"], suffixes=("_L", "_P"))

    merged = merged[(merged[f"{col}_P"] > 0) & (merged[f"{col}_L"] > 0)]
    if merged.empty:
        return pd.DataFrame()

    # Rev/EPS Change
    merged["Change"] = merged[f"{col}_L"] / merged[f"{col}_P"] - 1

    # Price Change
    merged["Price_Change"] = merged["Current Price_L"] / merged["Current Price_P"] - 1

    # Clean inf/nan
    merged["Change"] = merged["Change"].replace([np.inf, -np.inf], 0).fillna(0)
    merged["Price_Change"] = merged["Price_Change"].replace([np.inf, -np.inf], 0).fillna(0)

    # Round to 4 decimals
    merged["Change"] = merged["Change"].round(4)
    merged["Price_Change"] = merged["Price_Change"].round(4)

    merged["#Count"] = merged["#Count_L"]

    return merged[["YAHOO Ticker", "FIN Yr", "Change", "Price_Change", "#Count"]]

# ================= IDENTIFY Y1 & Y2 =================
fy_sorted = sorted(df_all["FIN Yr"].unique())
Y1 = fy_sorted[0]
Y2 = fy_sorted[1]

YMAP = {Y1: "Y1", Y2: "Y2"}

# ================= BUILD SHEETS =================
tp_master = []
rev_master = []
eps_master = []

# ---------- TARGET PRICE ----------
for period in PERIODS:
    for col, label in [
        ("TP(Min)", "TP Min Revision"),
        ("TP(Avg)", "TP Avg Revision"),
        ("TP(Max)", "TP Max Revision"),
    ]:

        header = f"{period} {label}"
        df_out = compute_tp_revision(df_all, col, period)

        if df_out.empty:
            continue

        # Upward
        df_up = df_out.sort_values("Change", ascending=False).head(TOP_N).reset_index(drop=True)
        df_up.insert(0, "Rank", df_up.index + 1)
        df_up.insert(0, "Direction", "Upward")
        df_up.insert(0, "Header", header)

        # Downward
        df_down = df_out.sort_values("Change", ascending=True).head(TOP_N).reset_index(drop=True)
        df_down.insert(0, "Rank", df_down.index + 1)
        df_down.insert(0, "Direction", "Downward")
        df_down.insert(0, "Header", header)

        tp_master.append(df_up)
        tp_master.append(df_down)

# ---------- REVENUE ----------
for period in PERIODS:
    for fin_year in [Y1, Y2]:
        y_label = YMAP[fin_year]

        for col, label in [
            ("Rev.(Min)", "Rev. Min Revision"),
            ("Rev.(Avg)", "Rev. Avg Revision"),
            ("Rev.(Max)", "Rev. Max Revision"),
        ]:

            header = f"{period} {y_label} {label}"
            df_out = compute_revision(df_all, col, period, fin_year)

            if df_out.empty:
                continue

            df_up = df_out.sort_values("Change", ascending=False).head(TOP_N).reset_index(drop=True)
            df_up.insert(0, "Rank", df_up.index + 1)
            df_up.insert(0, "Direction", "Upward")
            df_up.insert(0, "Header", header)

            df_down = df_out.sort_values("Change", ascending=True).head(TOP_N).reset_index(drop=True)
            df_down.insert(0, "Rank", df_down.index + 1)
            df_down.insert(0, "Direction", "Downward")
            df_down.insert(0, "Header", header)

            rev_master.append(df_up)
            rev_master.append(df_down)

# ---------- EPS ----------
for period in PERIODS:
    for fin_year in [Y1, Y2]:
        y_label = YMAP[fin_year]

        for col, label in [
            ("EPS(Min)", "EPS Min Revision"),
            ("EPS(Avg)", "EPS Avg Revision"),
            ("EPS(Max)", "EPS Max Revision"),
        ]:

            header = f"{period} {y_label} {label}"
            df_out = compute_revision(df_all, col, period, fin_year)

            if df_out.empty:
                continue

            df_up = df_out.sort_values("Change", ascending=False).head(TOP_N).reset_index(drop=True)
            df_up.insert(0, "Rank", df_up.index + 1)
            df_up.insert(0, "Direction", "Upward")
            df_up.insert(0, "Header", header)

            df_down = df_out.sort_values("Change", ascending=True).head(TOP_N).reset_index(drop=True)
            df_down.insert(0, "Rank", df_down.index + 1)
            df_down.insert(0, "Direction", "Downward")
            df_down.insert(0, "Header", header)

            eps_master.append(df_up)
            eps_master.append(df_down)

# ================= WRITE TO EXCEL =================
with pd.ExcelWriter(output_file, engine="xlsxwriter") as writer:
    pd.concat(tp_master, ignore_index=True).to_excel(writer, sheet_name="TrendAnalysis", index=False)
    pd.concat(rev_master, ignore_index=True).to_excel(writer, sheet_name="RevenueTrendAnalysis", index=False)
    pd.concat(eps_master, ignore_index=True).to_excel(writer, sheet_name="EPSTrendAnalysis", index=False)

print(f"\nExcel file created:\n{output_file}")


Excel file created:
C:\MyDocuments\03Business\05ResearchAndAnalysis\StockInvestments\ResearchReports\YahooConsensus\20260213_YahooConsensusTrendAnalysis.xlsx
